# Berkeley AUV gate YOLO training (Colab)

Same configuration as `intro-perception-train`: pretrained **yolo26n.pt**, one class (`gate`), **30 epochs**, **640 px**, seed 0.

1. *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
2. When the upload button appears, choose **`generated.zip`** (the zipped `data/generated` folder: `train/`, `val/`, `test/`, `dataset.yaml`).
3. At the end the browser downloads **`yolo_results.zip`** with `best.pt`, `predictions_yolo.jsonl` (test-split predictions in the project's format), the training curves and the test metrics.


In [ ]:
!pip -q install ultralytics==8.4.104
import ultralytics, torch
print(ultralytics.__version__, 'CUDA:', torch.cuda.is_available())


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose generated.zip
!rm -rf /content/dataset && mkdir -p /content/dataset && unzip -q generated.zip -d /content/dataset
# The zip may or may not have a top-level folder; find dataset.yaml either way.
import glob, os
DATA_YAML = sorted(glob.glob('/content/dataset/**/dataset.yaml', recursive=True))[0]
DATA_DIR = os.path.dirname(DATA_YAML)
for split in ('train', 'val', 'test'):
    print(split, len(os.listdir(f'{DATA_DIR}/{split}/images')), 'images')


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
model.train(data=DATA_YAML, epochs=30, imgsz=640, seed=0, deterministic=True,
            project='/content/runs', name='gate', exist_ok=True)


In [ ]:
# Held-out test split with Ultralytics' own metrics (for reference).
best = YOLO('/content/runs/gate/weights/best.pt')
test_metrics = best.val(data=DATA_YAML, split='test', imgsz=640, project='/content/runs', name='test', exist_ok=True)
print(test_metrics.results_dict)


In [ ]:
# Project-format predictions for intro-perception-evaluate: one record per test image,
# highest-confidence box, same conversion as intro_perception/yolo.py.
import json, time, cv2
def estimate(result, shape):
    boxes = result.boxes
    if boxes is None or len(boxes.conf) == 0:
        return {'visible': False, 'confidence': 0.0, 'center_x': 0.5, 'center_y': 0.5,
                'box_x': 0.0, 'box_y': 0.0, 'box_width': 0.0, 'box_height': 0.0}
    i = int(boxes.conf.argmax())
    x1, y1, x2, y2 = boxes.xyxy[i].tolist(); h, w = shape[:2]
    clamp = lambda v: max(0.0, min(1.0, v))
    bx, by = clamp(x1 / w), clamp(y1 / h)
    return {'visible': True, 'confidence': float(boxes.conf[i]),
            'center_x': clamp((x1 + x2) / 2 / w), 'center_y': clamp((y1 + y2) / 2 / h),
            'box_x': bx, 'box_y': by,
            'box_width': min(clamp((x2 - x1) / w), 1 - bx), 'box_height': min(clamp((y2 - y1) / h), 1 - by)}

def predict_all(device, path):
    images = sorted(glob.glob(f'{DATA_DIR}/test/images/*'))
    best.predict(cv2.imread(images[0]), device=device, verbose=False)  # warm-up
    with open(path, 'w') as out:
        for image_path in images:
            frame = cv2.imread(image_path)
            started = time.perf_counter()
            result = best.predict(frame, device=device, verbose=False)[0]
            elapsed = time.perf_counter() - started
            stem = os.path.splitext(os.path.basename(image_path))[0]
            out.write(json.dumps({'frame_id': stem, 'image': os.path.basename(image_path),
                                  'inference_sec': elapsed, **estimate(result, frame.shape)}) + '\n')
    times = [json.loads(l)['inference_sec'] for l in open(path)]
    print(device, f'{len(times) / sum(times):.1f} FPS over {len(times)} images')

predict_all(0 if torch.cuda.is_available() else 'cpu', '/content/predictions_yolo.jsonl')
predict_all('cpu', '/content/predictions_yolo_cpu.jsonl')   # same machine class as the classical FPS


In [ ]:
import shutil
os.makedirs('/content/yolo_results', exist_ok=True)
for path in ['/content/runs/gate/weights/best.pt', '/content/runs/gate/results.csv',
             '/content/runs/gate/results.png', '/content/runs/gate/args.yaml',
             '/content/predictions_yolo.jsonl', '/content/predictions_yolo_cpu.jsonl']:
    if os.path.exists(path):
        shutil.copy(path, '/content/yolo_results/')
json.dump({k: float(v) for k, v in test_metrics.results_dict.items()},
          open('/content/yolo_results/ultralytics_test_metrics.json', 'w'), indent=2)
shutil.make_archive('/content/yolo_results', 'zip', '/content/yolo_results')
files.download('/content/yolo_results.zip')
